# Phase 5 - NLLB-200 + MorphBPE Kapampangan->Filipino fine-tune (Bible-primary refresh)

**You run this in Colab. Claude Code wrote it and cannot run it.**

**2026-09-03 data refresh.** The training set is now **~3,590 pairs**, built
mostly from the **PLOC Kapampangan<->Tagalog Bible corpus** (the primary
downstream dataset named in the thesis proposal) plus supplementary
conversational / story / news material. Two held-out test sets:

- **`test_bible`** (329) - whole held-out Bible **chapters** (leakage-safe).
  The in-domain headline number.
- **`test_ood`** (70) - five whole native-authored stories + `gold_v1`
  sentences. The register-transfer number (train is ~71% religious).

The embedding recipe is unchanged from the warm-start rev: each new
6,080-row encoder embedding is **warm-started** from the mean of NLLB's own
sub-token embeddings for that token string (`vocab.json`), everything else
frozen, `tie_weights()` never called after the swap (Phase 4). Checkpoints
are selected on **dev loss**; generation eval (chrF++/BLEU, beam 4) runs
once at the end on both test sets. **3 seeds** per trained condition.

| condition | encoder source tokenizer | init |
|---|---|---|
| `nllb_zeroshot` | NLLB SentencePiece, 256K, pretrained | - (no training, reference) |
| `morphbpe` | hard-constrained MorphBPE @ 6,080, Kapampangan-trained | warm-start `nn.Embedding(6080,1024)` |
| `penalty8` | weighted MorphBPE (penalty 8) @ 6,080, Kapampangan-trained | warm-start `nn.Embedding(6080,1024)` |
| `unigram6080` | Unigram-LM (NLLB's algorithm) trained on this project's Kapampangan corpus @ 6,080 | warm-start `nn.Embedding(6080,1024)` |

Fair headline: `morphbpe` / `penalty8` **vs `unigram6080`** - same vocab,
corpus, init recipe; only the subword algorithm differs.

### How to run
1. `Runtime -> Change runtime type -> T4 GPU`.
2. **Delete any old `phase5-results.json`** in the Colab file panel (stale
   runs from a previous bundle would be skipped otherwise).
3. Upload the bundle (6 files from `experiments/nllb_finetune_v1/data/bundle/`):
   `train.jsonl`, `dev.jsonl`, `test_bible.jsonl`, `test_ood.jsonl`,
   `meta.json`, `vocab.json`.
4. `Runtime -> Run all`. **~4-6 h** (zero-shot + 3 conditions x 3 seeds) -
   plan for **two Colab sessions**. Results write incrementally after every
   run and a re-run **skips finished runs**, so on a disconnect just run all
   again to resume. For a faster first pass set `"seeds": [0, 1]` in
   `meta.json` before uploading. `batch` is **8** (T4-safe - Bible verses run
   ~40% longer than the old set); raise to 16 in `meta.json` if the runtime
   has headroom.
5. Download `phase5-results.json` and send it back.

**Data note.** The bundle contains the PLOC Bible corpus (**provenance via a
groupmate; believed PLOC/DLSU LTL / LGPL but UNCONFIRMED - redistribution
rights unresolved**), PLD-derived pairs (rights unresolved), native-authored
stories (author = you), `gold_v1`, and Gemini AI-generated sentences.
Uploaded to your Colab runtime only; nothing is published. Confirm you are
comfortable with the Bible corpus rights before uploading.

In [ ]:
%pip install -q -U "transformers>=4.44,<5" sentencepiece sacremoses sacrebleu
import json, os, random, time
import numpy as np
import torch, torch.nn as nn
import transformers, sacrebleu
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
print("transformers", transformers.__version__, "| torch", torch.__version__, "| sacrebleu", sacrebleu.__version__)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if DEVICE == "cuda":
    p = torch.cuda.get_device_properties(0)
    print("GPU:", p.name, round(p.total_memory/1024**3, 1), "GiB")
else:
    print("WARNING: no GPU - this will be very slow.")

In [ ]:
# --- load the bundle ---
def read_jsonl(path):
    with open(path, encoding="utf-8") as f:
        return [json.loads(l) for l in f if l.strip()]

for fn in ["train.jsonl", "dev.jsonl", "test_bible.jsonl", "test_ood.jsonl", "meta.json", "vocab.json"]:
    assert os.path.exists(fn), f"missing {fn} - upload the bundle files first"

META = json.load(open("meta.json", encoding="utf-8"))
VOCAB = json.load(open("vocab.json", encoding="utf-8"))          # {condition: [id-ordered strings]}
TRAIN = read_jsonl("train.jsonl"); DEV = read_jsonl("dev.jsonl")
TEST_BIBLE = read_jsonl("test_bible.jsonl"); TEST_OOD = read_jsonl("test_ood.jsonl")
TEST_SETS = {"test_bible": TEST_BIBLE, "test_ood": TEST_OOD}
print(f"train {len(TRAIN)} | dev {len(DEV)} | test_bible {len(TEST_BIBLE)} | test_ood {len(TEST_OOD)}")
print("conditions:", META["conditions"], "| seeds:", META["seeds"])

TGT_LANG = META["nllb"]["target_lang"]
NATIVE_SRC_LANG = META["nllb"]["native_baseline_source_lang"]
SRC_VOCAB = META["source_vocab_size_morphbpe"]    # 6080
SRC_PAD = META["source_pad_id_morphbpe"]          # 0
MODEL_NAME = META["nllb"]["model"]
HP = META["training"]["hyperparams"]
SEEDS = META["seeds"]
SWAP_CONDITIONS = set(META["conditions"])         # morphbpe, penalty8, unigram6080

RESULTS_PATH = "phase5-results.json"
results = json.load(open(RESULTS_PATH)) if os.path.exists(RESULTS_PATH) else {}
def save_results():
    json.dump(results, open(RESULTS_PATH, "w"), indent=2, ensure_ascii=False)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.tgt_lang = TGT_LANG
TGT_BOS = tokenizer.convert_tokens_to_ids(TGT_LANG)
PAD = tokenizer.pad_token_id

def fresh_model():
    m = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME, dtype=torch.float32)
    m.config.use_cache = False
    return m

def src_ids_for(condition, rec):
    if condition == "morphbpe":    return rec["morphbpe_ids"]
    if condition == "penalty8":    return rec["penalty8_ids"]
    if condition == "unigram6080": return rec["unigram_ids"]
    tokenizer.src_lang = NATIVE_SRC_LANG                      # nllb_zeroshot
    return tokenizer(rec["pam_text"], add_special_tokens=True)["input_ids"]

def label_ids(rec):
    return tokenizer(text_target=rec["fil_text"], add_special_tokens=True)["input_ids"]

def src_pad_for(condition):
    return SRC_PAD if condition in SWAP_CONDITIONS else PAD

def collate(batch, condition):
    src = [src_ids_for(condition, r) for r in batch]
    lab = [label_ids(r) for r in batch]
    sm = max(len(x) for x in src); lm = max(len(x) for x in lab); pad = src_pad_for(condition)
    ii = torch.full((len(batch), sm), pad, dtype=torch.long)
    am = torch.zeros((len(batch), sm), dtype=torch.long)
    lb = torch.full((len(batch), lm), -100, dtype=torch.long)
    for i,(s,l) in enumerate(zip(src, lab)):
        ii[i,:len(s)] = torch.tensor(s); am[i,:len(s)] = 1
        lb[i,:len(l)] = torch.tensor(l)
    return ii.to(DEVICE), am.to(DEVICE), lb.to(DEVICE)

In [ ]:
# --- warm-started encoder-embedding swap (Phase 4 recipe) ---
# map this project's special ids (pad0 unk1 bos2 eos3) onto NLLB's rows
NLLB_SPECIAL_ROW = {0: PAD, 1: tokenizer.unk_token_id, 2: tokenizer.bos_token_id, 3: tokenizer.eos_token_id}

def warm_start_embedding(condition, shared_weight):
    d = shared_weight.size(1)
    emb = nn.Embedding(SRC_VOCAB, d, padding_idx=SRC_PAD)
    strings = VOCAB[condition]
    with torch.no_grad():
        emb.weight.normal_(0.0, d ** -0.5)                    # fallback for empty lookups
        for i, s in enumerate(strings):
            if i in NLLB_SPECIAL_ROW:
                emb.weight[i] = shared_weight[NLLB_SPECIAL_ROW[i]].cpu()
                continue
            sub = tokenizer(s, add_special_tokens=False)["input_ids"]
            if sub:
                emb.weight[i] = shared_weight[sub].mean(0).cpu()
        emb.weight[SRC_PAD].zero_()
    return emb

def prepare_model(condition, seed):
    torch.manual_seed(seed); np.random.seed(seed); random.seed(seed)
    m = fresh_model()
    core = m.model
    for p in m.parameters():
        p.requires_grad_(False)
    emb = warm_start_embedding(condition, core.shared.weight.detach())
    emb.weight.requires_grad_(True)
    core.encoder.embed_tokens = emb          # encoder ONLY; never call tie_weights() now
    m.to(DEVICE)
    trainable = [p for p in m.parameters() if p.requires_grad]
    assert len(trainable) == 1 and trainable[0] is emb.weight
    return m, emb

In [ ]:
BATCH = HP["batch"]; EPOCHS = HP["epochs"]; PATIENCE = HP["patience"]; LR = HP["lr"]
GEN_KW = dict(num_beams=4, max_new_tokens=HP.get("max_new_tokens", 128), forced_bos_token_id=TGT_BOS)

def batched(seq, n):
    for i in range(0, len(seq), n):
        yield seq[i:i+n]

@torch.no_grad()
def dev_loss(m, condition):
    m.eval(); tot = 0.0; nb = 0
    for chunk in batched(DEV, BATCH):
        ii, am, lb = collate(chunk, condition)
        tot += float(m(input_ids=ii, attention_mask=am, labels=lb).loss); nb += 1
    return tot / nb

@torch.no_grad()
def eval_gen(m, data, condition):
    m.eval(); hyps = []; refs = [r["fil_text"] for r in data]
    for chunk in batched(data, BATCH):
        src = [src_ids_for(condition, r) for r in chunk]
        sm = max(len(x) for x in src); pad = src_pad_for(condition)
        ii = torch.full((len(chunk), sm), pad, dtype=torch.long)
        am = torch.zeros((len(chunk), sm), dtype=torch.long)
        for i,s in enumerate(src):
            ii[i,:len(s)] = torch.tensor(s); am[i,:len(s)] = 1
        out = m.generate(input_ids=ii.to(DEVICE), attention_mask=am.to(DEVICE), **GEN_KW)
        hyps += tokenizer.batch_decode(out, skip_special_tokens=True)
    return {"bleu": round(sacrebleu.corpus_bleu(hyps, [refs]).score, 2),
            "chrf": round(sacrebleu.corpus_chrf(hyps, [refs], word_order=2).score, 2)}

def eval_all_tests(m, condition):
    return {name: eval_gen(m, data, condition) for name, data in TEST_SETS.items()}

def train_one(condition, seed):
    key = f"{condition}/seed{seed}"
    if key in results:
        print("skip (done):", key); return
    t0 = time.time()
    m, emb = prepare_model(condition, seed)
    opt = torch.optim.AdamW([emb.weight], lr=LR)
    order = list(range(len(TRAIN)))
    best_loss = float("inf"); best_state = emb.weight.detach().clone(); best_ep = 0; bad = 0
    for ep in range(1, EPOCHS+1):
        m.train(); random.Random(1000+seed*97+ep).shuffle(order)
        tr = 0.0; nb = 0
        for idx in batched(order, BATCH):
            ii, am, lb = collate([TRAIN[i] for i in idx], condition)
            loss = m(input_ids=ii, attention_mask=am, labels=lb).loss
            loss.backward(); opt.step(); opt.zero_grad()
            tr += float(loss.detach()); nb += 1
        dl = dev_loss(m, condition)
        print(f"  {key} ep{ep:02d}  train {tr/nb:.3f}  dev-loss {dl:.3f}")
        if dl < best_loss - 1e-3:
            best_loss = dl; best_state = emb.weight.detach().clone(); best_ep = ep; bad = 0
        else:
            bad += 1
            if bad >= PATIENCE:
                print("  early stop"); break
    with torch.no_grad():
        emb.weight.copy_(best_state)
    dev_score = eval_gen(m, DEV, condition)
    test_scores = eval_all_tests(m, condition)
    results[key] = {"condition": condition, "seed": seed, "best_epoch": best_ep,
                    "best_dev_loss": round(best_loss, 3),
                    "dev": dev_score, **test_scores,
                    "minutes": round((time.time()-t0)/60, 1)}
    save_results()
    print(f"  -> {key}  DEV {dev_score}  test_bible {test_scores['test_bible']}  "
          f"test_ood {test_scores['test_ood']}  ({results[key]['minutes']} min)")
    del m, emb; torch.cuda.empty_cache()

In [ ]:
# --- zero-shot reference (no training), once ---
if "nllb_zeroshot" not in results:
    m = fresh_model().to(DEVICE)
    results["nllb_zeroshot"] = {"condition": "nllb_zeroshot",
                                "dev": eval_gen(m, DEV, "nllb_zeroshot"),
                                **eval_all_tests(m, "nllb_zeroshot")}
    save_results(); del m; torch.cuda.empty_cache()
    print("nllb_zeroshot", results["nllb_zeroshot"])

# --- the trained runs ---
for condition in META["conditions"]:
    for seed in SEEDS:
        train_one(condition, seed)

In [ ]:
# --- summary + download ---
from statistics import mean, pstdev
TEST_NAMES = list(TEST_SETS)          # ["test_bible", "test_ood"]

agg = {}
for condition in META["conditions"]:
    runs = [results[f"{condition}/seed{s}"] for s in SEEDS if f"{condition}/seed{s}" in results]
    if not runs:
        continue
    row = {"n_seeds": len(runs)}
    for tn in TEST_NAMES:
        tc = [r[tn]["chrf"] for r in runs]; tb = [r[tn]["bleu"] for r in runs]
        row[tn] = {"chrf_mean": round(mean(tc), 2), "chrf_std": round(pstdev(tc), 2),
                   "bleu_mean": round(mean(tb), 2), "bleu_std": round(pstdev(tb), 2)}
    agg[condition] = row
if "nllb_zeroshot" in results:
    agg["nllb_zeroshot"] = {tn: results["nllb_zeroshot"][tn] for tn in TEST_NAMES}

summary = {"aggregate": agg, "per_run": results,
           "meta": {"model": MODEL_NAME, "train": len(TRAIN), "dev": len(DEV),
                    "test_bible": len(TEST_BIBLE), "test_ood": len(TEST_OOD),
                    "warm_start": True, "select_on": "dev_loss", "seeds": SEEDS,
                    "hyperparams": HP, "all_silver": True,
                    "bible_rights": "UNRESOLVED (groupmate; believed PLOC/DLSU LTL / LGPL)",
                    "fair_headline": "morphbpe / penalty8 vs unigram6080"}}
json.dump(summary, open("phase5-results.json", "w"), indent=2, ensure_ascii=False)
print(json.dumps(agg, indent=2))
try:
    from google.colab import files
    files.download("phase5-results.json")
except Exception as e:
    print("download manually:", e)

## Reading the result

Two numbers per condition:

- **`test_bible`** (in-domain) is the headline. Compare each
  `aggregate.<cond>.test_bible.chrf_mean` to
  `aggregate.nllb_zeroshot.test_bible.chrf`. With ~3.6k mostly-religious
  training pairs the trained conditions should now clearly beat zero-shot
  in-domain; if they don't, the frozen-encoder recipe still isn't coping
  with the new tokenisation granularity -> LoRA on the encoder (a separate
  condition) or accept the null result.
- **`test_ood`** (register transfer) shows how much a Bible-heavy encoder
  embedding generalises to conversational text. Expect it lower than
  `test_bible`; the interesting question is whether the tokenizer ranking
  is the *same* across the two.

**Fair comparison:** `morphbpe` / `penalty8` vs `unigram6080` - same vocab,
corpus, init recipe; only the subword algorithm differs. With 3 seeds you
have `chrf_std` / `bleu_std`: a gap smaller than the across-seed std is not
a real effect (a legitimate finding either way for a low-resource study).

Send `phase5-results.json` back -> `experiments/nllb_finetune_v1/reports/`.